# Silver Layer — Cleaning & Transformation (Azure Databricks / PySpark)

Reads the raw AdventureWorks CSVs from the **bronze** container, standardises column names and types, cleans and enriches each table, and writes Parquet files to the **silver** container for Synapse to query.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Connect to the storage account with a service principal.
# Credentials are read from a Databricks secret scope (e.g. backed by Azure Key Vault)
# so that no secrets are stored in the notebook.
storage_account = "a1projectstorage"
secret_scope = "cri-key-vault"

client_id = dbutils.secrets.get(scope=secret_scope, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=secret_scope, key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope=secret_scope, key="tenant-id")

configs = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

for key, value in configs.items():
    spark.conf.set(key, value)

### Reading Data from Containers

In [0]:
# Define function to read CSV files
def read_csv(file_path):
    return spark.read.format("csv") \
        .option("header", True) \
        .option("inferSchema", True) \
        .load(file_path)

# Dictionary to store file paths
file_paths = {
    "Calendar": "abfss://bronze@a1projectstorage.dfs.core.windows.net/AdventureWorks_Calendar",
    "Customers": "abfss://bronze@a1projectstorage.dfs.core.windows.net/AdventureWorks_Customers",
    "Product_Categories": "abfss://bronze@a1projectstorage.dfs.core.windows.net/AdventureWorks_Product_Categories",
    "Products": "abfss://bronze@a1projectstorage.dfs.core.windows.net/AdventureWorks_Products",
    "Returns": "abfss://bronze@a1projectstorage.dfs.core.windows.net/AdventureWorks_Returns",
    "Sales": "abfss://bronze@a1projectstorage.dfs.core.windows.net/AdventureWorks_Sales*",
    "Territories": "abfss://bronze@a1projectstorage.dfs.core.windows.net/AdventureWorks_Territories",
    "Product_Subcategories": "abfss://bronze@a1projectstorage.dfs.core.windows.net/Product_Subcategories"
}

# Read all files into a dictionary
dfs = {name: read_csv(path) for name, path in file_paths.items()}

# Assign to variables if needed
df_cal = dfs["Calendar"]
df_cus = dfs["Customers"]
df_procat = dfs["Product_Categories"]
df_pro = dfs["Products"]
df_ret = dfs["Returns"]
df_sales = dfs["Sales"]
df_ter = dfs["Territories"]
df_subcat = dfs["Product_Subcategories"]

# Show schema for verification
for name, df in dfs.items():
    print(f"Schema for {name}:")
    df.printSchema()

Schema for Calendar:
root
 |-- Date: date (nullable = true)

Schema for Customers:
root
 |-- CustomerKey: integer (nullable = true)
 |-- Prefix: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EmailAddress: string (nullable = true)
 |-- AnnualIncome: string (nullable = true)
 |-- TotalChildren: integer (nullable = true)
 |-- EducationLevel: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- HomeOwner: string (nullable = true)

Schema for Product_Categories:
root
 |-- ProductCategoryKey: integer (nullable = true)
 |-- CategoryName: string (nullable = true)

Schema for Products:
root
 |-- ProductKey: integer (nullable = true)
 |-- ProductSubcategoryKey: integer (nullable = true)
 |-- ProductSKU: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- ModelName: string 

# Transformation

### Calendar Data & Analysis

In [0]:
# Convert Date column to DateType and extract Year, Month, and Day from Date
df_cal = df_cal.withColumn("Date", col("Date").cast(DateType())) \
               .withColumn("Year", year(col("Date"))) \
               .withColumn("Month", month(col("Date"))) \
               .withColumn("Day", dayofmonth(col("Date")))

# Rename all column names to lowercase
df_cal = df_cal.select([col(c).alias(c.lower()) for c in df_cal.columns])

display(df_cal.limit(5))

date,year,month,day
2015-01-01,2015,1,1
2015-01-02,2015,1,2
2015-01-03,2015,1,3
2015-01-04,2015,1,4
2015-01-05,2015,1,5


In [0]:
# Count occurrences per month
df_monthly_counts = df_cal.groupBy("month").count().orderBy(col("count").desc())
display(df_monthly_counts)

month,count
1,93
3,93
5,93
6,90
4,90
2,85
12,62
8,62
7,62
10,62


Databricks visualization. Run in Databricks to view.

### Customers Data & Analysis

In [0]:
# Convert Date column to DateType
df_cus = df_cus.withColumn("birthdate", col("birthdate").cast(DateType()))

# Converting "AnnualIncome" column to double type & removing $ sign from the column
df_cus = df_cus.withColumn("annualincome", regexp_replace(col("annualincome"), "[$,]", "").cast(DoubleType()))

# Standardize all category names to lowercase and rename all column names to lowercase
df_cus = df_cus.select([lower(col(c)).alias(c.lower()) if df_cus.schema[c].dataType.simpleString() == "string" else col(c) for c in df_cus.columns])

# Add full name column by combining first name and last name column and place this column after prefix
df_cus = df_cus.withColumn("fullname", concat_ws(" ", col("firstname"), col("lastname")))
cols = df_cus.columns
prefix_index = cols.index("prefix")
cols.insert(prefix_index + 1, cols.pop(cols.index("fullname")))
df_cus = df_cus.select(cols)

display(df_cus)

In [0]:
# Define a function to display distribution counts
def display_distribution(df, column):
    display(df.groupBy(column).count().orderBy("count", ascending=False))

# Gender Distribution
display_distribution(df_cus, "gender")

# Marital Status Distribution
display_distribution(df_cus, "maritalstatus")

# Education Level Distribution
display_distribution(df_cus, "educationlevel")

# Most Common Occupation
display_distribution(df_cus, "occupation")

gender,count
m,9126
f,8892
na,130


Databricks visualization. Run in Databricks to view.

maritalstatus,count
m,9817
s,8331


Databricks visualization. Run in Databricks to view.

educationlevel,count
bachelors,5261
partial college,4966
high school,3241
graduate degree,3125
partial high school,1555


Databricks visualization. Run in Databricks to view.

occupation,count
professional,5424
skilled manual,4501
management,3011
clerical,2859
manual,2353


Databricks visualization. Run in Databricks to view.

#### Products & sub products data sets analysis

In [0]:
# Standardize all category names to lowercase and rename all column names to lowercase
df_procat = df_procat.select([lower(col(c)).alias(c.lower()) if df_procat.schema[c].dataType.simpleString() == "string" else col(c).alias(c.lower()) for c in df_procat.columns])
df_subcat = df_subcat.select([lower(col(c)).alias(c.lower()) if df_subcat.schema[c].dataType.simpleString() == "string" else col(c).alias(c.lower()) for c in df_subcat.columns])

# Display cleaned datasets
display(df_procat)
display(df_subcat)

productcategorykey,categoryname
1,bikes
2,components
3,clothing
4,accessories


productsubcategorykey,subcategoryname,productcategorykey
1,mountain bikes,1
2,road bikes,1
3,touring bikes,1
4,handlebars,2
5,bottom brackets,2
6,brakes,2
7,chains,2
8,cranksets,2
9,derailleurs,2
10,forks,2


In [0]:
# Count subcategories per category
subcategories_per_category = df_subcat.groupBy("productcategorykey").count().orderBy("count", ascending=False)
display(subcategories_per_category)

# Find category with the most subcategories
category_most_subcategories = df_procat.join(df_subcat, "productcategorykey") \
                                       .groupBy("categoryname").count() \
                                       .orderBy("count", ascending=False)
display(category_most_subcategories)

productcategorykey,count
2,14
4,12
3,8
1,3


categoryname,count
components,14
accessories,12
clothing,8
bikes,3


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, trim, lower

# Convert column names to lowercase and remove leading/trailing spaces in text fields
df_pro = df_pro.select([trim(lower(col(c))).alias(c.lower()) if df_pro.schema[c].dataType.simpleString() == "string" else col(c).alias(c.lower()) for c in df_pro.columns])

# Ensure foreign key relationships with Product_Subcategories
df_pro = df_pro.join(df_subcat, "productsubcategorykey", "inner")

# Remove duplicates (if any)
df_pro = df_pro.dropDuplicates(["productkey"])

# Convert numeric columns to appropriate data types
df_pro = df_pro.withColumn("productcost", col("productcost").cast("double")) \
               .withColumn("productprice", col("productprice").cast("double"))

display(df_pro)

In [0]:
# How many unique products exist?
unique_products = df_pro.select(countDistinct("productkey").alias("unique_products"))
display(unique_products)

# What is the average product price?
average_product_price = df_pro.select(round(avg("productprice"), 2).alias("average_product_price"))
display(average_product_price)

# What is the price distribution of products?
price_distribution = df_pro.select("productprice")
display(price_distribution)

# Which product subcategory has the most products?
subcategory_most_products = df_pro.groupBy("productsubcategorykey") \
                                  .count() \
                                  .orderBy(desc("count")) \
                                  .limit(5) \
                                  .join(df_subcat, "productsubcategorykey") \
                                  .select("subcategoryname", "count")
display(subcategory_most_products)

# Which colors are the most common for products?
most_common_color = df_pro.groupBy("productcolor") \
                          .count() \
                          .orderBy(desc("count")) \
                          .limit(5)
display(most_common_color)

unique_products
293


average_product_price
714.44


productprice
63.5
1003.91
175.49
32.6
39.14
34.99
1700.99
74.99
8.99
769.49


Databricks visualization. Run in Databricks to view.

subcategoryname,count
mountain bikes,32
road bikes,43
touring bikes,22
mountain frames,28
road frames,31


Databricks visualization. Run in Databricks to view.

productcolor,count
black,88
na,50
red,37
silver,36
yellow,36


Databricks visualization. Run in Databricks to view.

#### Sales Data Transformation and Analysis

In [0]:
# Standardize column names and clean data
df_sales = df_sales.select(
    col("OrderDate").cast(DateType()).alias("orderdate"),
    col("StockDate").cast(DateType()).alias("stockdate"),
    col("OrderNumber").alias("ordernumber"),
    col("ProductKey").alias("productkey"),
    col("CustomerKey").alias("customerkey"),
    col("TerritoryKey").alias("territorykey"),
    col("OrderLineItem").cast(IntegerType()).alias("orderlineitem"),
    col("OrderQuantity").cast(IntegerType()).alias("orderquantity")
)

# Ensure product, customer, and territory keys exist in respective tables
df_sales = df_sales.join(df_pro, "productkey", "inner") \
                   .join(df_cus, "customerkey", "inner") \
                   .join(df_ter, df_sales["territorykey"] == df_ter["salesterritorykey"], "inner") \
                   .drop("salesterritorykey")

# Remove duplicates (if any)
df_sales = df_sales.dropDuplicates(["ordernumber", "orderlineitem"])

# Standardize column names
df_sales = df_sales.select([lower(col(c)).alias(c) if df_sales.schema[c].dataType.simpleString() == "string" else col(c) for c in df_sales.columns])

# Show transformed data
display(df_sales)

In [0]:
# Calculate total sales revenue
total_revenue = df_sales.withColumn("totalsales", col("orderquantity") * col("productprice")) \
                        .select(round(sum("totalsales"), 2).alias("totalrevenue"))
display(total_revenue)

# Find best-selling product by revenue
best_selling_products = df_sales.withColumn("totalsales", col("orderquantity") * col("productprice")) \
                                .groupBy("productname") \
                                .agg(round(sum("totalsales"), 2).alias("revenue")) \
                                .orderBy(desc("revenue")) \
                                .limit(5)
display(best_selling_products)

# Find top customers by total spending
top_customers = df_sales.withColumn("totalspending", col("orderquantity") * col("productprice")) \
                        .groupBy("customerkey", "firstname", "lastname") \
                        .agg(round(sum("totalspending"), 2).alias("totalspending")) \
                        .orderBy(desc("totalspending")) \
                        .limit(5)
display(top_customers)

# Find highest revenue-generating sales territory
top_sales_territories = df_sales.withColumn("totalsales", col("orderquantity") * col("productprice")) \
                                .groupBy("region", "country") \
                                .agg(round(sum("totalsales"), 2).alias("revenue")) \
                                .orderBy(desc("revenue"))
display(top_sales_territories)

# Analyze peak sales months
peak_sales_months = df_sales.withColumn("totalsales", col("orderquantity") * col("productprice")) \
                            .groupBy(month("orderdate").alias("month")) \
                            .agg(round(sum("totalsales"), 2).alias("revenue")) \
                            .orderBy(desc("revenue"))
display(peak_sales_months)

totalrevenue
2.491458682E7


productname,revenue
"mountain-200 black, 46",1241753.51
"mountain-200 black, 42",1233557.12
"mountain-200 silver, 38",1213851.89
"mountain-200 silver, 46",1182780.59
"mountain-200 black, 38",1165936.88


Databricks visualization. Run in Databricks to view.

customerkey,firstname,lastname,totalspending
11433,maurice,shan,12407.95
11439,janet,munoz,12015.4
11241,lisa,cai,11330.45
11417,lacey,zheng,11085.75
11420,jordan,turner,11022.4


Databricks visualization. Run in Databricks to view.

region,country,revenue
australia,australia,7416456.2
southwest,united states,4822794.7
northwest,united states,3095074.47
united kingdom,united kingdom,2902562.09
germany,germany,2524679.97
france,france,2362643.32
canada,canada,1769245.81
southeast,united states,11585.62
northeast,united states,6401.57
central,united states,3143.06


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

month,revenue
6,3030800.79
5,2973293.15
4,2676135.18
3,2563994.11
2,2345630.33
1,2292117.05
12,2199070.33
11,1460524.2
10,1434097.65
8,1340646.2


Databricks visualization. Run in Databricks to view.

#### Sales Data Transformation and Analysis

In [0]:
# Standardize column names and clean data
df_ret = df_ret.select(
    col("ReturnDate").cast(DateType()).alias("returndate"),
    col("TerritoryKey").alias("territorykey"),
    col("ProductKey").alias("productkey"),
    col("ReturnQuantity").cast(IntegerType()).alias("returnquantity")
)

# Ensure product and territory keys exist in respective tables
df_ret = df_ret.join(df_pro, "productkey", "inner") \
               .join(df_ter, df_ret["territorykey"] == df_ter["salesterritorykey"], "inner") \
               .drop("salesterritorykey")

# Remove duplicates (if any)
df_ret = df_ret.dropDuplicates(["returndate", "productkey", "territorykey"])

# Standardize column names
df_ret = df_ret.select([lower(col(c)).alias(c) if df_ret.schema[c].dataType == StringType() else col(c) for c in df_ret.columns])

# Show transformed data
display(df_ret)

In [0]:
# Count total returned products
total_returns = df_ret.agg(sum("returnquantity").alias("totalreturns"))
display(total_returns)

# Find most returned products
most_returned_products = df_ret.groupBy("productname") \
                               .agg(sum("returnquantity").alias("totalreturns")) \
                               .orderBy(desc("totalreturns")) \
                               .limit(5)
display(most_returned_products)

# Find sales territories with most returns
sales_territories_most_returns = df_ret.groupBy("region", "country") \
                                       .agg(sum("returnquantity").alias("totalreturns")) \
                                       .orderBy(desc("totalreturns"))
display(sales_territories_most_returns)

# Find peak months for returns
peak_months_returns = df_ret.groupBy(month("returndate").alias("month")) \
                            .agg(sum("returnquantity").alias("totalreturns")) \
                            .orderBy(desc("totalreturns"))
display(peak_months_returns)

totalreturns
1828


productname,totalreturns
water bottle - 30 oz.,155
patch kit/8 patches,95
mountain tire tube,93
mountain bottle cage,77
"sport-100 helmet, red",70


Databricks visualization. Run in Databricks to view.

region,country,totalreturns
australia,australia,404
southwest,united states,362
northwest,united states,270
canada,canada,238
united kingdom,united kingdom,204
france,france,186
germany,germany,163
southeast,united states,1


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

month,totalreturns
5,190
4,183
6,179
3,177
12,176
1,170
2,166
10,148
11,141
8,126


Databricks visualization. Run in Databricks to view.

### Write Data to Silver Layer

In [0]:
#Write data to silver container

storage_path = "abfss://silver@a1projectstorage.dfs.core.windows.net/"

df_cal.write.mode("overwrite").format("parquet").save(f"{storage_path}Calenders")
df_cus.write.mode("overwrite").format("parquet").save(f"{storage_path}Customers")
df_pro.write.mode("overwrite").format("parquet").save(f"{storage_path}Products")
df_procat.write.mode("overwrite").format("parquet").save(f"{storage_path}Product_Categories")
df_ret.write.mode("overwrite").format("parquet").save(f"{storage_path}Returns")
df_sales.write.mode("overwrite").format("parquet").save(f"{storage_path}Sales")
df_subcat.write.mode("overwrite").format("parquet").save(f"{storage_path}Sub_Categories")
df_ter.write.mode("overwrite").format("parquet").save(f"{storage_path}Territories")

print("All datasets have been successfully stored in Parquet format in the Silver Layer!")

All datasets have been successfully stored in Parquet format in the Silver Layer!
